#### Imports y config

In [34]:
from pathlib import Path
import duckdb
import pandas as pd
import glob
import re
import os
from datetime import datetime

PROJECT_ROOT = str(Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent)
os.chdir(PROJECT_ROOT)

con = duckdb.connect("data/db/casitas.duckdb")
print(f"✅ Directorio: {os.getcwd()}")

# ── LIMPIAR CSV DE LINKS ─────────────────────────────
df_links = pd.read_csv("data/raw/links_viviendas.csv", header=0)
df_links.columns = ["url", "extra"]
df_links["url"] = df_links["url"].str.strip()

# Normalizar URLs — quitar parámetros UTM para deduplicar mejor
def normalizar_url(url):
    # Quitar todo lo que viene después de ? para comparar solo el inmueble
    return str(url).split("?")[0].split("/foto/")[0].rstrip("/")

df_links["url_norm"] = df_links["url"].apply(normalizar_url)

antes = len(df_links)
df_links = df_links.drop_duplicates(subset="url_norm", keep="first")
df_links = df_links.drop(columns=["url_norm"])
despues = len(df_links)

print(f"Antes: {antes} | Después: {despues} | Eliminados: {antes - despues}")

# Guardar limpio
df_links.to_csv("data/raw/links_viviendas.csv", index=False, header=False)
print(f"✅ CSV limpio guardado con {despues} links únicos")


# ── EXTRACTOR WHATSAPP ───────────────────────────────
DOMINIOS = [
    "idealista", "fotocasa", "pisos.com",
    "yaencontre", "tecnocasa", "habitaclia"
]

def es_valido(url):
    url = str(url).lower()
    if "obra-nueva" in url: return False
    if "terreno" in url: return False
    if "alquilar" in url or "alquiler" in url: return False
    return True

def extraer_links_whatsapp(ruta_chat):
    with open(ruta_chat, "r", encoding="utf-8") as f:
        texto = f.read()
    patron = r'https?://[^\s\]>)"]+'
    todos = re.findall(patron, texto)
    inmobiliarios = [
        url for url in todos
        if any(d in url for d in DOMINIOS) and es_valido(url)
    ]
    return list(dict.fromkeys(inmobiliarios))

def agregar_links_nuevos(links_nuevos):
    if not links_nuevos:
        print("✅ No hay links nuevos")
        return

    df_existentes = pd.read_csv("data/raw/links_viviendas.csv", header=0)
    df_existentes.columns = ["url", "extra"]
    urls_existentes = set(df_existentes["url"].str.strip().tolist())

    realmente_nuevos = [u for u in links_nuevos if u not in urls_existentes]

    if not realmente_nuevos:
        print("✅ Todos los links ya existen")
        return

    df_nuevos = pd.DataFrame({"url": realmente_nuevos, "extra": ""})
    df_actualizado = pd.concat([df_existentes, df_nuevos], ignore_index=True)
    df_actualizado.to_csv("data/raw/links_viviendas.csv", index=False, header=False)

    print(f"✅ {len(realmente_nuevos)} links nuevos agregados")
    print(f"Total: {len(df_actualizado)}")

ruta_chat = "data/raw/_chat.txt"
if os.path.exists(ruta_chat):
    links = extraer_links_whatsapp(ruta_chat)
    print(f"Links válidos encontrados: {len(links)}")
    agregar_links_nuevos(links)
else:
    print("⚠️ No hay _chat.txt en data/raw/")

✅ Directorio: <project-root>
Antes: 191 | Después: 183 | Eliminados: 8
✅ CSV limpio guardado con 183 links únicos
Links válidos encontrados: 177
✅ 10 links nuevos agregados
Total: 192


In [35]:
print("""
⚠️  PAUSA — antes de continuar:
    1. Abre terminal
    2. cd <project-root>
    3. source casitas/bin/activate
    4. python src/pipeline.py
    5. Espera que termine
    6. Vuelve aquí y continúa
""")


⚠️  PAUSA — antes de continuar:
    1. Abre terminal
    2. cd <project-root>
    3. source casitas/bin/activate
    4. python src/pipeline.py
    5. Espera que termine
    6. Vuelve aquí y continúa



In [36]:
# ── CARGAR CSVs CON DUCKDB ───────────────────────────
archivos = glob.glob("data/raw/*_scraped_*.csv")
print(f"CSVs encontrados: {len(archivos)}")
for a in sorted(archivos):
    print(f"  - {a}")

con.execute("""
CREATE OR REPLACE TABLE viviendas_raw AS
SELECT *
FROM read_csv_auto('data/raw/*_scraped_*.csv')
""")

df_raw = con.execute("SELECT * FROM viviendas_raw").df()

print(f"\nTotal registros: {len(df_raw)}")
print(df_raw["plataforma"].value_counts())
print(df_raw["estado_anuncio"].value_counts())

CSVs encontrados: 11
  - data/raw/fotocasa_scraped_20260520_1131.csv
  - data/raw/fotocasa_scraped_20260524_1129.csv
  - data/raw/idealista_scraped_20260520_1131.csv
  - data/raw/idealista_scraped_20260520_1657.csv
  - data/raw/idealista_scraped_20260524_1129.csv
  - data/raw/pisos_scraped_20260520_1131.csv
  - data/raw/pisos_scraped_20260524_1129.csv
  - data/raw/tecnocasa_scraped_20260520_1131.csv
  - data/raw/tecnocasa_scraped_20260524_1129.csv
  - data/raw/yaencontre_scraped_20260520_1131.csv
  - data/raw/yaencontre_scraped_20260524_1129.csv

Total registros: 297
plataforma
idealista     208
fotocasa       35
yaencontre     35
pisos          12
tecnocasa       7
Name: count, dtype: int64
estado_anuncio
activo                   225
dado de baja              70
alquiler — descartado      2
Name: count, dtype: int64


In [37]:
# ── LIMPIAR CON DUCKDB ───────────────────────────────
con.execute("""
CREATE OR REPLACE TABLE viviendas_limpias AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY url
               ORDER BY url
           ) AS rn
    FROM viviendas_raw
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TABLE viviendas_activas AS
SELECT *
FROM viviendas_limpias
WHERE estado_anuncio = 'activo'
""")

df_raw     = con.execute("SELECT * FROM viviendas_limpias").df()
df_activos = con.execute("SELECT * FROM viviendas_activas").df()

# Convertir columnas numéricas
cols_numericas = ["precio", "m2", "habitaciones", "baños", "año"]
for col in cols_numericas:
    df_raw[col]     = pd.to_numeric(df_raw[col], errors="coerce").astype("Int64")
    df_activos[col] = pd.to_numeric(df_activos[col], errors="coerce").astype("Int64")

# Limpiar comentarios
for df in [df_raw, df_activos]:
    df["comentario"] = df["comentario"].str.replace(
        r'Al guardar, aceptas nuestras condiciones.*', '', regex=True
    ).str.strip()
    df["comentario"] = df["comentario"].str.replace(
        r'\s*Leer comentario completo\s*', '', regex=True
    ).str.strip()

print(f"Limpios: {len(df_raw)} | Activos: {len(df_activos)}")
print(f"\nNulos en activos:")
print(df_activos[cols_numericas].isnull().sum())

Limpios: 188 | Activos: 136

Nulos en activos:
precio          26
m2              30
habitaciones    38
baños           36
año             94
dtype: int64


In [38]:
# ── DETECTAR SITUACIÓN ───────────────────────────────
def detectar_situacion(comentario, titulo):
    texto = (str(comentario) + " " + str(titulo)).lower()

    if any(k in texto for k in [
        "alquilado", "con inquilinos", "vivienda alquilada",
        "actualmente alquilad", "renta mensual", "contrato de arrendamiento"
    ]):
        return "alquilado"

    if any(k in texto for k in [
        "okupa", "ocupado", "posible ocupación", "no se puede visitar"
    ]):
        return "ocupado"

    if any(k in texto for k in [
        "cesión de remate", "subasta", "sin posesión",
        "no admite hipoteca", "procedimiento judicial"
    ]):
        return "subasta/remate"

    if any(k in texto for k in [
        "npl", "nuda propiedad", "usufructo"
    ]):
        return "nuda propiedad/npl"

    if any(k in texto for k in [
        "cambio de uso", "no es vivienda"
    ]):
        return "local"

    return "libre"

df_activos["situacion"] = df_activos.apply(
    lambda r: detectar_situacion(r["comentario"], r["titulo"]), axis=1
)

print("Situación detectada:")
print(df_activos["situacion"].value_counts())

Situación detectada:
situacion
libre                 125
alquilado               4
local                   3
ocupado                 2
subasta/remate          1
nuda propiedad/npl      1
Name: count, dtype: int64


In [39]:
# ── GUARDAR ──────────────────────────────────────────
os.makedirs("data/interim", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

df_raw.to_csv(f"data/interim/todas_viviendas_{timestamp}.csv", index=False)
df_activos.to_csv(f"data/processed/activos_{timestamp}.csv", index=False)

print(f"✅ Guardado:")
print(f"   data/interim/todas_viviendas_{timestamp}.csv — {len(df_raw)} registros")
print(f"   data/processed/activos_{timestamp}.csv — {len(df_activos)} registros")

✅ Guardado:
   data/interim/todas_viviendas_20260524_1341.csv — 188 registros
   data/processed/activos_20260524_1341.csv — 136 registros


In [40]:
# ── VERIFICAR DISTRIBUCIÓN DE LINKS ─────────────────
df_links = pd.read_csv("data/raw/links_viviendas.csv", header=0)
df_links.columns = ["url", "extra"]
df_links["url"] = df_links["url"].str.strip()

def detectar_plataforma(url):
    for nombre, dominio in {
        "idealista":  "idealista",
        "fotocasa":   "fotocasa",
        "pisos":      "pisos.com",
        "habitaclia": "habitaclia",
        "tecnocasa":  "tecnocasa",
        "yaencontre": "yaencontre",
    }.items():
        if dominio in str(url):
            return nombre
    return "otro"

df_links["plataforma"] = df_links["url"].apply(detectar_plataforma)

print(f"Total links en CSV: {len(df_links)}")
print(df_links["plataforma"].value_counts())

Total links en CSV: 191
plataforma
idealista     120
fotocasa       28
yaencontre     23
pisos           8
habitaclia      7
tecnocasa       5
Name: count, dtype: int64


In [41]:
df_links = pd.read_csv("data/raw/links_viviendas.csv", header=0)
df_links.columns = ["url", "extra"]
df_links["url"] = df_links["url"].str.strip()

total = len(df_links)
unicos = df_links["url"].nunique()
duplicados = total - unicos

print(f"Total links:    {total}")
print(f"Únicos:         {unicos}")
print(f"Duplicados:     {duplicados}")

if duplicados > 0:
    print("\nURLs duplicadas:")
    print(df_links[df_links.duplicated(subset="url", keep=False)]["url"].tolist())

Total links:    191
Únicos:         191
Duplicados:     0
